# 05 · Treinar QLoRA CPC — Qwen3-8B, contexto 1024

Este notebook executa **treinamento completo do piloto**: 160 famílias de treino, 20 de validação, três épocas e 30 atualizações do otimizador. As 20 famílias de teste permanecem reservadas. Não é necessário repetir 00, 02, 03 ou o diagnóstico 04b aprovado.

O modelo usa NF4 em 4 bits, cálculo FP16, LoRA `r=8` em `q_proj`/`v_proj`, uma família por microbatch e acumulação de 16. O contexto máximo é 1024. A perda usa apenas a resposta CPC e o marcador final. Os títulos e resumos permanecem em português/inglês.

**Antes de executar:** atualize no Drive o conjunto completo do notebook 05: notebook, script de treino, perfil de treino, helper e `SHA256SUMS.txt`, além das dependências e dados já presentes. Abra esta cópia **05** no Colab, escolha GPU e execute as células em ordem. Não carregue E5 ou outro modelo simultaneamente. O cache dos pesos fica em `/content`; adaptadores, checkpoints e relatórios ficam no Drive.

O sucesso do diagnóstico 04b demonstra viabilidade de um passo na sessão testada. A perda 5,895 daquele diagnóstico não mede qualidade final. Este treino seleciona um adaptador pela perda de validação; ainda não avalia os seis campos de extração nem as métricas finais do estudo. O CPC é um proxy de supervisão, e 17/20 famílias de validação têm rótulos CPC integrais ausentes no treino.


## 1. Montar o Drive e informar os caminhos

Mantenha o relatório completo do diagnóstico aprovado no caminho abaixo. O resumo copiado para o chat não substitui esse JSON. Para a primeira execução, deixe `RETOMAR_CHECKPOINT` vazio. Depois de uma interrupção, informe nele o **diretório de checkpoint completo** mostrado pelo relatório anterior; não informe o diretório do adaptador selecionado.


In [ ]:
from pathlib import Path
from google.colab import drive
import os

os.environ['HF_HOME'] = '/content/hf_cache_llm_eidt_br'
os.environ['USE_TF'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')
PASTA_DRIVE = '/content/drive/MyDrive/Replicação LLM EIDT IC' # @param {type:'string'}
DIAGNOSTICO_APROVADO = 'resultados/etapa_1/piloto_br_200/qlora_teste_20261104T014632833741Z.json' # @param {type:'string'}
RETOMAR_CHECKPOINT = '' # @param {type:'string'}

raiz = Path(PASTA_DRIVE).expanduser().resolve(strict=True)
ponto_de_montagem = Path('/content/drive').resolve(strict=True)
if raiz == ponto_de_montagem or not raiz.is_relative_to(ponto_de_montagem) or not raiz.is_dir():
    raise ValueError('Selecione a pasta do projeto dentro do Drive montado.')

def caminho_no_projeto(relativo):
    parte = Path(relativo)
    if not relativo or parte.is_absolute() or '..' in parte.parts:
        raise ValueError(f'O caminho deve ser relativo à pasta do projeto: {relativo!r}')
    alvo = (raiz / parte).resolve()
    if alvo == raiz or not alvo.is_relative_to(raiz):
        raise ValueError('O caminho informado sai da pasta do projeto.')
    return alvo

perfil_relativo = 'config/perfil_treino_qlora_1024.json'
script_relativo = 'scripts/treinar_qlora_cpc.py'
print('Pasta selecionada:', raiz)
print('Modo:', 'retomada explícita' if RETOMAR_CHECKPOINT.strip() else 'novo treinamento')


## 2. Conferir versões, dados e diagnóstico aprovado

A conferência abaixo interrompe a execução se houver arquivos trocados ou incompletos. Ela confere as assinaturas do pacote e das entradas congeladas, o relatório real de capacidade e as contagens da auditoria. O arquivo da partição de teste não é aberto.


In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
from importlib import metadata
import json, math, re, subprocess, sys, tempfile

# Preenchidos na publicação deste pacote. Não copie hashes de outra versão.
SHA256_PERFIL_TREINO = '8931f1847423f6ad9cf77f7f1ae2023e982c41697ba5fb75b56f5782fc8bbec8'
SHA256_SCRIPT_TREINO = 'ada0a21cac6ec36e9d2d8df053dc0a1416dccfd0fe11674046c328d02727de25'
ID_PERFIL = 'br_patstat_200_qwen3_8b_qlora_nf4_fp16_train_v1'
ID_CAPACIDADE = 'br_patstat_200_qwen3_8b_qlora_nf4_fp16_smoke_v2_ctx1024'

indice = raiz / 'SHA256SUMS.txt'
assinaturas = {}
for linha in indice.read_text(encoding='utf-8-sig').splitlines():
    item = re.fullmatch(r'([0-9a-f]{64})  (.+)', linha)
    if item:
        digest, nome = item.groups()
        if nome in assinaturas:
            raise ValueError(f'Entrada duplicada em SHA256SUMS.txt: {nome}')
        assinaturas[nome] = digest

conferidos = {}
def ler_conferido(relativo, esperado):
    if not re.fullmatch(r'[0-9a-f]{64}', str(esperado)):
        raise ValueError(f'Assinatura ausente ou inválida: {relativo}')
    arquivo = caminho_no_projeto(relativo)
    if not arquivo.is_file():
        raise FileNotFoundError(f'Copie este arquivo para o Drive: {relativo}')
    conteudo = arquivo.read_bytes()
    digest = sha256(conteudo).hexdigest()
    if digest != esperado:
        raise ValueError(f'Arquivo de outra versão ou alterado: {relativo}. Atualize o pacote completo do 05.')
    conferidos[relativo] = digest
    return conteudo

for nome in ('requirements.txt', 'config/experimento.json', perfil_relativo, script_relativo,
             'scripts/testar_qlora_gpu.py', 'config/perfil_qlora_teste_1024.json'):
    ler_conferido(nome, assinaturas.get(nome))
    print('Conferido:', nome)
perfil = json.loads(ler_conferido(perfil_relativo, SHA256_PERFIL_TREINO))
ler_conferido(script_relativo, SHA256_SCRIPT_TREINO)
if perfil['id'] != ID_PERFIL or perfil['treino'] != {
    'max_length': 1024, 'microbatch': 1, 'acumulacao': 16, 'epocas': 3, 'checkpoint_cada_passos': 5
}:
    raise ValueError('Use o perfil de treino 05 publicado: contexto 1024, três épocas e acumulação 16.')
origem = perfil['origem']
entradas = {}
for chave in ('config', 'auditoria_preprocessamento', 'treino', 'validacao', 'perfil_capacidade', 'helper'):
    entradas[chave] = ler_conferido(origem[chave], origem['sha256_' + chave])
base = json.loads(entradas['config'])
auditoria = json.loads(entradas['auditoria_preprocessamento'])
capacidade = json.loads(entradas['perfil_capacidade'])
if auditoria['particoes'] != {'treino': 160, 'validacao': 20, 'teste': 20} or auditoria['total_familias'] != 200:
    raise ValueError('As contagens da auditoria não correspondem ao piloto 160/20/20.')
if auditoria['estado'] != 'entradas_preparadas_sem_execucao_de_modelo' or auditoria['campos_vazios'] != 0:
    raise ValueError('A auditoria de pré-processamento não está no estado esperado.')
if auditoria['sha256_config'] != origem['sha256_config']:
    raise ValueError('A auditoria foi criada para outra configuração.')
for chave in ('treino', 'validacao'):
    if auditoria['sha256_arquivos'][Path(origem[chave]).name] != origem['sha256_' + chave]:
        raise ValueError(f'A auditoria não confirma a assinatura de {chave}.')
if perfil['modelo'] != capacidade['modelo'] or perfil['modelo'] != base['modelos']['qwen']:
    # A configuração principal pode trazer campos adicionais além de id/revisão.
    if any(perfil['modelo'].get(k) != capacidade['modelo'].get(k) or
           perfil['modelo'].get(k) != base['modelos']['qwen'].get(k) for k in ('id', 'revisao')):
        raise ValueError('Modelo ou revisão diverge do protocolo e do diagnóstico.')
if capacidade['id'] != ID_CAPACIDADE or capacidade['teste']['max_length'] != 1024:
    raise ValueError('O perfil de capacidade precisa ser a versão v2 com contexto 1024.')
for chave in ('quantizacao', 'lora'):
    if perfil[chave] != capacidade[chave]:
        raise ValueError(f'O perfil de treinamento mudou o item validado: {chave}.')

caminho_diagnostico = caminho_no_projeto(DIAGNOSTICO_APROVADO.strip())
conteudo_diagnostico = caminho_diagnostico.read_bytes()
diagnostico = json.loads(conteudo_diagnostico)
if diagnostico.get('estado') != 'teste_qlora_aprovado' or diagnostico.get('etapa') != 'concluido':
    raise ValueError('É necessário o relatório completo de um diagnóstico QLoRA aprovado.')
if diagnostico.get('sha256_perfil') != origem['sha256_perfil_capacidade'] or diagnostico.get('perfil') != capacidade:
    raise ValueError('O diagnóstico não confirma este perfil completo de capacidade. Não use somente o resumo do chat.')
if diagnostico.get('erro') or diagnostico.get('etapa_falha'):
    raise ValueError('O relatório de diagnóstico registra uma falha.')
if RETOMAR_CHECKPOINT.strip() and not caminho_no_projeto(RETOMAR_CHECKPOINT.strip()).is_dir():
    raise FileNotFoundError('O diretório de checkpoint informado não existe no Drive.')

inicio_notebook = datetime.now(timezone.utc)
pasta_relatorios = caminho_no_projeto(perfil['resultados'])
pasta_relatorios.mkdir(parents=True, exist_ok=True)
relatorio_preparacao = pasta_relatorios / f'preparacao_treino_notebook_{inicio_notebook:%Y%m%dT%H%M%S%fZ}.json'
preparacao = {
    'inicio_utc': inicio_notebook.isoformat(), 'estado': 'arquivos_conferidos',
    'perfil': perfil['id'], 'sha256_arquivos': conferidos,
    'diagnostico': DIAGNOSTICO_APROVADO.strip(),
    'sha256_diagnostico': sha256(conteudo_diagnostico).hexdigest(),
    'retomar_checkpoint': RETOMAR_CHECKPOINT.strip() or None,
    'cache_temporario': os.environ['HF_HOME'],
    'particoes_conforme_auditoria': auditoria['particoes'],
}
def salvar_preparacao():
    preparacao['atualizado_utc'] = datetime.now(timezone.utc).isoformat()
    temporario = relatorio_preparacao.with_suffix('.tmp')
    temporario.write_text(json.dumps(preparacao, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    temporario.replace(relatorio_preparacao)
salvar_preparacao()
arquivo_requisitos = raiz / 'requirements.txt'
print('\nTREINO QLORA 1024 — conferência aprovada')
print('Famílias:', auditoria['particoes'], '· total:', auditoria['total_familias'])
print('Épocas: 3 · microbatch: 1 · acumulação: 16 · atualizações planejadas: 30')
print('Modelo:', perfil['modelo']['id'], '· revisão:', perfil['modelo']['revisao'])
print('Capacidade aprovada:', caminho_diagnostico)
print('Preparação:', relatorio_preparacao)


## 3. Instalar as dependências preservando o ambiente do Colab

A célula mantém as versões nativas de PyTorch e NumPy e instala somente os requisitos fixados. O treinamento será iniciado em um processo separado; este notebook não importa PyTorch no processo principal. Aguarde a mensagem de dependências conferidas antes de prosseguir.


In [ ]:
try:
    versoes_protegidas = {nome: metadata.version(nome) for nome in ("torch", "numpy")}
    preparacao["versoes_protegidas"] = versoes_protegidas
    desejadas = {}
    for linha in arquivo_requisitos.read_text(encoding="utf-8-sig").splitlines():
        linha = linha.split("#", 1)[0].strip()
        if linha:
            if not re.fullmatch(r"[A-Za-z0-9_.-]+==[^\s]+", linha):
                raise ValueError("Os requisitos devem usar versões exatas no formato pacote==versão.")
            nome, versao = linha.split("==", 1)
            desejadas[nome] = versao
    divergentes = []
    for nome, esperada in desejadas.items():
        try:
            atual = metadata.version(nome)
        except metadata.PackageNotFoundError:
            atual = None
        if atual != esperada:
            divergentes.append(nome)
    preparacao["pacotes_que_requeriam_instalacao"] = divergentes
    if divergentes:
        with tempfile.TemporaryDirectory(dir="/content") as pasta_temporaria:
            restricoes = Path(pasta_temporaria) / "base_colab.txt"
            plano_pip = Path(pasta_temporaria) / "operacoes_pip.json"
            restricoes.write_text(
                "".join(f"{nome}=={versao}\n" for nome, versao in versoes_protegidas.items()),
                encoding="utf-8",
            )
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                 "--progress-bar", "off", "-c", str(restricoes), "-r", str(arquivo_requisitos),
                 "--report", str(plano_pip)],
                check=True,
            )
            operacoes = json.loads(plano_pip.read_text(encoding="utf-8"))["install"]
            preparacao["pacotes_alterados"] = sorted(item["metadata"]["name"] for item in operacoes)
    else:
        preparacao["pacotes_alterados"] = []
    for nome, anterior in versoes_protegidas.items():
        if metadata.version(nome) != anterior:
            raise RuntimeError(f"A versão protegida de {nome} mudou. Interrompa e confira o ambiente.")
    preparacao["versoes_requisitos"] = {nome: metadata.version(nome) for nome in desejadas}
    if preparacao["versoes_requisitos"] != desejadas:
        raise RuntimeError("Alguma dependência ainda diverge da versão fixada.")
    preparacao["estado"] = "dependencias_conferidas"
    print("Dependências conferidas. PyTorch e NumPy nativos preservados.")
except Exception as erro:
    preparacao["estado"] = "falha_na_instalacao"
    preparacao["tipo_erro"] = type(erro).__name__
    raise
finally:
    salvar_preparacao()


## 4. Conferir GPU, RAM e armazenamento

Os pesos originais ocupam aproximadamente 16 GB antes da quantização no carregamento. São exigidos 22 GiB livres se o cache ainda não contiver todos os arquivos de pesos; com o cache completo, a reserva mínima local é de 4 GiB. O Drive receberá os adaptadores, checkpoints e relatórios. Esta verificação informa os recursos disponíveis, mas a viabilidade das três épocas será confirmada pela execução real.


In [ ]:
if preparacao.get('estado') != 'dependencias_conferidas':
    raise RuntimeError('Conclua a instalação e conferência das dependências antes de continuar.')
codigo_recursos = r"""
import json, os, platform, shutil
from pathlib import Path
import torch
ram = {}
for linha in Path('/proc/meminfo').read_text().splitlines():
    chave, _, valor = linha.partition(':')
    if chave in ('MemTotal', 'MemAvailable'):
        ram[chave] = round(int(valor.split()[0]) / 1024**2, 2)
saida = {'python': platform.python_version(), 'torch': str(torch.__version__),
         'cuda_pytorch': torch.version.cuda, 'cuda_disponivel': torch.cuda.is_available(),
         'disco_livre_gib': round(shutil.disk_usage('/content').free / 1024**3, 2), 'ram_gib': ram}
if saida['cuda_disponivel']:
    livre, total = torch.cuda.mem_get_info(0)
    saida.update(gpu=torch.cuda.get_device_name(0), capacidade_cuda=list(torch.cuda.get_device_capability(0)),
                 vram_livre_gib=round(livre/1024**3, 3), vram_total_gib=round(total/1024**3, 3))
print(json.dumps(saida))
"""
try:
    verificador = subprocess.run([sys.executable, '-c', codigo_recursos], capture_output=True, text=True, check=True)
    recursos = json.loads(verificador.stdout.strip().splitlines()[-1])
    snapshot = (Path(os.environ['HF_HOME']) / 'hub' / 'models--Qwen--Qwen3-8B' /
                'snapshots' / perfil['modelo']['revisao'])
    indice_pesos = snapshot / 'model.safetensors.index.json'
    cache_completo = False
    if indice_pesos.is_file():
        fragmentos = set(json.loads(indice_pesos.read_text())['weight_map'].values())
        cache_completo = bool(fragmentos) and all(
            Path(nome).name == nome and (snapshot / nome).is_file() and (snapshot / nome).stat().st_size > 0
            for nome in fragmentos
        )
    recursos['cache_pesos_completo'] = cache_completo
    recursos['disco_minimo_gib'] = 4 if cache_completo else 22
    preparacao['recursos'] = recursos
    print(json.dumps(recursos, ensure_ascii=False, indent=2))
    if not recursos['cuda_disponivel']:
        raise RuntimeError('Selecione GPU no tipo de ambiente de execução do Colab.')
    if recursos['capacidade_cuda'][0] < 6:
        raise RuntimeError('A GPU não atende à capacidade CUDA mínima para este perfil NF4.')
    if recursos['disco_livre_gib'] < recursos['disco_minimo_gib']:
        raise RuntimeError('O armazenamento local está abaixo da reserva necessária. Revise o espaço antes do treino.')
    preparacao['estado'] = 'pronto_para_treino'
    print('Ambiente preparado. A próxima célula inicia o treinamento de três épocas.')
except Exception as erro:
    preparacao['estado'] = 'recursos_requerem_atencao'
    preparacao['tipo_erro'] = type(erro).__name__
    if isinstance(erro, subprocess.CalledProcessError):
        print((erro.stderr or '')[-4000:])
    raise
finally:
    salvar_preparacao()


## 5. Executar o treinamento e selecionar o adaptador

Esta célula carrega o Qwen3-8B e executa as três épocas. A validação usa 20 famílias antes do treino e ao final de cada época. A referência inicial é registrada separadamente; entre as épocas 1, 2 e 3, será selecionada a menor perda média por token de CPC + EOS, mantendo a primeira época em caso de empate.

Os checkpoints são gravados a cada cinco atualizações e ao fim das épocas. Não há reutilização automática de adaptadores. Se a sessão cair, reabra este notebook, monte o Drive e preencha `RETOMAR_CHECKPOINT` com o último checkpoint **completo** apontado no relatório. Execute novamente desde a célula 1. O script confere a compatibilidade antes de retomar.

Não feche a sessão durante a gravação. Se houver falha, leia o resumo e preserve o relatório antes de tentar novamente. O tempo total inclui download, carregamento, avaliação e gravação; os 861 segundos do diagnóstico não servem como estimativa de uma época.


In [ ]:
if preparacao.get('estado') != 'pronto_para_treino':
    raise RuntimeError('Execute primeiro as conferências de arquivos, dependências e recursos.')
# Nova chamada tem relatório próprio, inclusive quando retoma a mesma execução.
chamada = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
saida_relativa = f"{perfil['resultados']}/chamada_treino_{chamada}.json"
relatorio_chamada = caminho_no_projeto(saida_relativa)
comando = [sys.executable, '-u', str(raiz / script_relativo),
           '--raiz', str(raiz), '--perfil', perfil_relativo,
           '--diagnostico', DIAGNOSTICO_APROVADO.strip(), '--saida_relatorio', saida_relativa]
if RETOMAR_CHECKPOINT.strip():
    comando.extend(['--retomar', RETOMAR_CHECKPOINT.strip()])
preparacao['relatorio_treino'] = saida_relativa
preparacao['estado'] = 'treino_em_execucao'
salvar_preparacao()
codigo_saida = None
interrompido = False
relatorio = None
print('Relatório desta chamada:', relatorio_chamada, flush=True)
try:
    processo = subprocess.run(comando, check=False)
    codigo_saida = processo.returncode
except KeyboardInterrupt:
    interrompido = True
    print('\nExecução interrompida. Confira o último checkpoint completo antes da retomada.')
finally:
    if relatorio_chamada.is_file():
        try:
            relatorio = json.loads(relatorio_chamada.read_text(encoding='utf-8'))
            campos = ('estado', 'passos_concluidos', 'epocas_concluidas', 'validacao_baseline_loss',
                      'melhor_epoca', 'melhor_validacao_loss', 'melhorou_sobre_baseline',
                      'delta_validacao_baseline', 'adaptador_selecionado',
                      'ultimo_checkpoint', 'pasta_execucao', 'pico_alocado_gib',
                      'pico_reservado_gib', 'duracao_segundos', 'erro')
            resumo = {nome: relatorio.get(nome) for nome in campos}
            print('\nRESUMO DO TREINAMENTO — copie este bloco')
            print(json.dumps(resumo, ensure_ascii=False, indent=2))
        except (ValueError, OSError) as erro_leitura:
            print('Não foi possível ler o relatório:', str(erro_leitura))
    else:
        print('O processo não chegou a criar o relatório. Preserve a mensagem de erro acima.')
    preparacao['estado'] = 'treino_concluido' if codigo_saida == 0 and relatorio and relatorio.get('estado') == 'treino_qlora_concluido' else 'treino_requer_atencao'
    preparacao['codigo_saida'] = codigo_saida
    preparacao['interrompido'] = interrompido
    salvar_preparacao()
    print('Relatório completo:', relatorio_chamada)
    print('Preparação do ambiente:', relatorio_preparacao)
if interrompido:
    raise RuntimeError('Treino interrompido. Use o último checkpoint completo para retomar.')
if codigo_saida != 0 or not relatorio or relatorio.get('estado') != 'treino_qlora_concluido':
    raise RuntimeError('O treinamento não foi concluído. Confira o resumo e o erro acima antes de repetir a execução.')
print('\nTreinamento concluído. Os arquivos de saída estão no Drive.')


## 6. O que guardar e enviar

Envie o bloco **RESUMO DO TREINAMENTO**, o caminho do relatório completo e eventuais mensagens de falha. O relatório, a preparação do ambiente, os checkpoints e o adaptador selecionado já são gravados no Drive. Preserve a pasta da execução e salve uma cópia deste notebook com as saídas para o registro do experimento.

Uma execução concluída deve registrar 30 atualizações e três épocas, além da perda inicial, da melhor época e do caminho do adaptador. A perda pode não melhorar; isso é um resultado do piloto, não um motivo para escolher outra partição ou repetir sementes até obter um número melhor. Ainda faltam a comparação com o modelo base, a avaliação reservada e a validação independente da extração. Este notebook não executa essas etapas.

Se não houver mais trabalho nesta sessão, desconecte e exclua o ambiente de execução para encerrar o uso da GPU. O cache em `/content` desaparecerá, mas os resultados salvos no Drive permanecerão.
